# Módulo F4 · Matemáticas para ML — Práctica guiada

**Caso:** *TiendaSur* quiere entender a sus clientes. Dirección pregunta:

1. ¿Qué clientes se parecen entre sí? *(distancias y similitud coseno)*
2. ¿Qué variables del dataset van juntas? *(covarianza)*
3. ¿Podemos "aprender" el gasto esperado en función de los pedidos con un modelo a mano? *(gradiente descendente)*
4. Si un cliente abre el email, ¿cuánto sube la probabilidad de que compre? *(Bayes)*

Todo se responde con NumPy, sin librerías de ML: la idea es ver la maquinaria funcionando a mano.

> **Ejecuta las celdas en orden.** Todas se pueden ejecutar en local o Colab sin descargar datos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 4.1 · Clientes como vectores: distancia y similitud

Cada cliente es un punto: `(gasto_mensual, nº_pedidos, ticket_medio)`. Cuanto más cerca, más parecidos.

In [ ]:
# 6 clientes de TiendaSur, 3 features
clientes = pd.DataFrame({
    "cliente":      ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo"],
    "gasto":        [120,  480,  90,  150, 510, 300],
    "pedidos":      [4,    20,   3,    5,   18,  10],
    "ticket_medio": [30,   24,   30,   30,  28,  30],
})
X = clientes[["gasto", "pedidos", "ticket_medio"]].to_numpy(float)
X

In [ ]:
def distancia_euclidea(a, b):
    # la distancia "de la vida real": la longitud del vector diferencia
    return np.linalg.norm(a - b)

print("Ana ↔ Elena :", round(distancia_euclidea(X[0], X[4]), 1))
print("Ana ↔ Carla :", round(distancia_euclidea(X[0], X[2]), 1))
print("Bruno ↔ Elena:", round(distancia_euclidea(X[1], X[4]), 1))

**Pregunta para pensar:** Bruno y Elena están muy lejos en euros pero son quienes más gastan. ¿Qué pasa si, en vez de euros con magnitud, uso el **coseno**?

La distancia mide magnitud; la similitud coseno mide *dirección*. Un cliente que gasta mucho en muchos pedidos con ticket medio parecido a otro "apunta" en la misma dirección aunque uno invierte 300 € y otro 600 €.

In [ ]:
def similitud_coseno(a, b):
    # 1 = misma dirección, 0 = perpendiculares, negativo = direcciones opuestas
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

for i, j in [(0, 2), (0, 4), (1, 4), (1, 5)]:
    print(f"{clientes.cliente[i]:>7} ↔ {clientes.cliente[j]:<7} coseno = {similitud_coseno(X[i], X[j]):.3f}")

## 4.2 · Covarianza: qué variables van juntas

La **matriz de covarianza** dice para cada par de variables "si una sube, ¿la otra sube también (o baja)?". Es la primera aproximación a relaciones, antes de modelar nada.

In [ ]:
# Covarianza entre las 3 features
cov = np.cov(X.T)
cov

In [ ]:
# Como es simétrica, la parte interesante es el triángulo superior
labels = ["gasto", "pedidos", "ticket_medio"]
for i in range(3):
    for j in range(i + 1, 3):
        print(f"cov({labels[i]:>12}, {labels[j]:<12}) = {cov[i, j]:8.2f}")

**Lectura:** covarianza positiva = van de la mano; negativa = una sube cuando la otra baja. En el módulo I1 esto servirá para reducir dimensiones (PCA) y aquí basta para responder "el gasto crece con el número de pedidos, pero el ticket medio casi ni se mueve: el motor del gasto es el **recurrente**, no el **caro**.

## 4.3 · Gradiente descendente desde cero: una recta que "aprende" el gasto

Modelo: `gasto ≈ w · pedidos + b`, con w (pendiente) y b (intercepto).

**Error:** cuadrático medio, `MSE(w, b) = media( (y - ŷ)² )`.

**Entrenar = buscar (w, b) que minimiza el MSE.** El gradiente lo hace por pasos:

In [ ]:
# Datos del ejemplo: nº de pedidos → gasto mensual (nuevo cliente hipotético)
pedidos_mensuales = np.array([ 2,  3,  4,  5,  6,  7,  8,  9, 10])
gasto_mensual     = np.array([45, 60, 80, 95,110,125,150,165,180])   # ≈ 15 € por pedido + 15 fijo

# Inicialización "a ciegas"
w, b = 0.0, 0.0
lr = 0.001

# Descenso de gradiente: 500 pasos
for paso in range(500):
    y_pred = w * pedidos_mensuales + b
    error  = y_pred - gasto_mensual          # el residuo = "cuánto me he equivocado"
    grad_w = 2 / len(gasto_mensual) * np.sum(error * pedidos_mensuales)
    grad_b = 2 / len(gasto_mensual) * np.sum(error)
    w -= lr * grad_w
    b -= lr * grad_b

print(f"w aprendido ≈ {w:.3f}  (el 'real' es ~15)")
print(f"b aprendido ≈ {b:.3f}  (el 'real' es ~15)")

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(pedidos_mensuales, gasto_mensual, s=40)
plt.plot(pedidos_mensuales, w * pedidos_mensuales + b, "r-",
         label=f"modelo aprendido: y ≈ {w:.2f}·x + {b:.2f}")
plt.legend(); plt.xlabel("pedidos/mes"); plt.ylabel("gasto/mes")
plt.title("La recta que el gradiente descendente encontró"); plt.show()

### ¿Qué pasa si cambias la tasa de aprendizaje? (lo prueba el ejercicio 3)

*Ejercicio en clase:* sube `lr` a 0.05 y corre de nuevo; luego bájala a 0.00005.
Con LR chico converge lento y se queda corto; con LR grande **oscila** — o se dispara. No es magia: la dirección del paso es fija, solo cambia el tamaño del paso.

## 4.4 · Comprobar el gradiente a mano con una derivada numérica

No hay que fiarse de fórmulas escritas por nosotros: se comprueban contra la definición numérica.

In [ ]:
def derivada_numerica(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

f  = lambda x: x**2
df = lambda x: 2*x                # derivada "a mano"
for x in [1.0, 4.0, 10.0]:
    print(f"x={x:5.1f}  analítica={df(x):7.4f}  numérica={derivada_numerica(f, x):7.4f}")

Si las dos columnas coinciden (≈), tu gradiente está bien escrito. **Este truco** salvaguarda la práctica de I5 (deep learning): cuando un gradiente manual falla, se compara con una versión numérica y se ve la diferencia.

## 4.5 · Bayes: actualizar P(compra | abre email)

Escenario de la campaña de email de TiendaSur:
- P(abre) = 0.10 — 1 de cada 10 clientes abre.
- P(compra | abre) = 0.40 — de los que abren, el 40 % compra (sí, es alto; es un ejemplo didáctico).
- P(compra) = 0.05 — 5 % de todos los clientes compra.

**Pregunta:** si un cliente *abre* el email, ¿cuánto sube la probabilidad de compra?

In [ ]:
P_abre        = 0.10
P_compra_dado_abre = 0.40
P_compra      = 0.05

# Ley de probabilidad total: P(compra) = P(compra|abre)·P(abre) + P(compra|no abre)·P(no abre)
P_no_abre         = 1 - P_abre
P_compra_dado_no  = (P_compra - P_compra_dado_abre * P_abre) / P_no_abre

# Bayes: ¿P(abre | compra)?
P_abre_dado_compra = P_compra_dado_abre * P_abre / P_compra

print(f"P(compra | no abre el email) ≈ {P_compra_dado_no:.3f}")
print(f"P(abre  | compra)            ≈ {P_abre_dado_compra:.3f}")

# La actualización que le importa al negocio:
print(f"P(compra | abre) = {P_compra_dado_abre:.0%} vs P(compra) = {P_compra:.0%}")
print(f"→ abre el email {P_compra_dado_abre / P_compra:.1f}x la probabilidad de comprar")

## 4.6 · Simulación: el examen de la fórmula

Bayes arriba se escribió con fórmulas. En una **simulación** de 50 000 clientes repetimos el proceso y **contamos**.

In [ ]:
N = 50_000
abre   = rng.random(N) < P_abre                       # 10 % abre
compra = np.where(abre,
                  rng.random(N) < P_compra_dado_abre,  # 40 % de los que abren
                  rng.random(N) < P_compra_dado_no)    # la probabilidad del resto

frac_compra_total      = compra.mean()
frac_abre_entre_compra = abre[compra].mean()
frac_compra_entre_abre = compra[abre].mean()

print(f"P(compra)               simulada = {frac_compra_total:.3f}  (fórmula: {P_compra:.3f})")
print(f"P(abre  | compra)       simulada = {frac_abre_entre_compra:.3f}  (fórmula: {P_abre_dado_compra:.3f})")
print(f"P(compra | abre)        simulada = {frac_compra_entre_abre:.3f}  (fórmula: {P_compra_dado_abre:.3f})")

Las tres columnas coinciden: fórmula e intuición cuadran.

---

## Ejercicios (5) — hazlo para la entrega

**E1.** Con la tabla `clientes` de la sección 4.1, encuentra, **por similitud coseno**, el vecino más parecido a cada cliente (excluye a sí mismo). Escribe también la distancia euclídea del mismo vecino y di una frase: ¿cambia el "más parecido" si usas distancia en vez de coseno? ¿Por qué?

**E2.** Añade la variable `edad` a `clientes` con valores `[25, 45, 19, 32, 50, 40]`. Recalcula la matriz de covarianza e interprétala en 3 frases: ¿qué variable crece con cuál, cuál casi se mueve sola?

**E3.** En la sección 4.3, ejecuta el mismo descenso con `lr = 0.05` y anota qué ocurre en los primeros 20 pasos (¿se estabiliza, oscila, diverge?). Después repite con `lr = 0.00005`. Entrega una gráfica del error (MSE) por paso para ambas tasas.

**E4.** Implementa `descenso(f, x0, lr, pasos)` con `derivada_numerica(f, x)` y comprueba que converge al mínimo de `f(x) = x² − 4x + 4`. ¿Cuál es ese mínimo y en qué valor de x?

**E5.** Con las probabilidades del escenario de email del notebook, imagina que la campaña mejora y P(compra | abre) sube a 0.55 manteniendo P(abre) = 0.10. Recalcula `P(compra | no abre)` con ley de probabilidad total y razona (una frase): ¿es razonable un caso donde esa probabilidad salga negativa? ¿Qué te diría eso de los supuestos?